# Benchmark III: a 123-qubit angle family that loads the real table exactly

**What this notebook establishes, and what it deliberately does not.**

It builds a family of `ANGLE_k` circuits for the dfthcsos submission at $brot = 10$ bits
per word and $J = 10$ words per circuit, and checks it with the competition's own
`scoreinfra.circuits`: the register contract, word recovery on all 3,976 addresses, the
phase, the workspace, and the reverse pass. It fits the 128-qubit cap with **five qubits
spare**, uses **eight** circuits for 75 angles, and reproduces all 298,200 angle words
bit-exactly.

It does **not** report a score. Scoring needs `FeMoCo76o113e.fcidump`, the 128,128,348-byte
packaged integral file that `scoreinfra/reference.py` pins by SHA-256 and refuses to
substitute. Without it `verify.py` stops before reading a single circuit. Every number
below is measured; none of them is a score, and the notebook says so where it matters
rather than in a footnote.

Three findings, in the order they arrived:

1. **Toffoli cost is set by the address tree, not by word width.** Measured 3,974
   Toffoli-like gates per circuit against a 3,975-node full tree, at every `(brot, J)`
   tried. Precision therefore costs almost nothing in `T` and a lot in qubits.
2. **That makes the batch the lever.** Dropping from 12 to 10 bits buys $J = 10$ instead
   of $J = 8$, which cuts the family from 10 circuits to 8 and its two-pass Toffoli count
   from 79,480 to **63,584**, a 19.98% reduction, for 4 extra qubits and 2 bits of precision.
3. **The 9-bit delta-compression idea is false on this table.** The Givens sign invariant
   `bit 9 == bit 8` holds on only **78.59%** of words, because this table's angles span
   $\theta/\pi \in [0, 2)$ rather than the $\pm\pi/2$ the argument assumed. Recorded as a
   falsification, with the measurement that kills it.

## 1. Locate the competition bundle

The circuits are checked by the bundle's own code, so the bundle must be mounted. A run
that silently used a local stand-in would be reporting on a different verifier, so this
cell stops if it cannot find one.

In [ ]:
import sys, pathlib

ROOTS = [pathlib.Path("/kaggle/input"), pathlib.Path.cwd(), pathlib.Path.cwd() / ".."]
bundle = None
for root in ROOTS:
    for cand in sorted(root.glob("**/verifiers/dfthcsos")) if root.is_dir() else []:
        bundle = cand.parent.parent
        break
    if bundle:
        break
assert bundle is not None, "competition bundle not found; mount the dfthcsos data first"

sys.path.insert(0, str(bundle / "verifiers"))
sys.path.insert(0, str(bundle / "kickmix"))
import numpy as np
import kickmix as km
from dfthcsos.scoreinfra import circuits as C

npz = next(p for p in sorted((bundle / "sample_submission" / "dfthcsos").glob("*.npz")))
with np.load(npz, allow_pickle=False) as z:
    words12 = np.array(z["angle_words"], dtype=np.int64)
print("bundle:", bundle.name, "| kickmix has mr:", hasattr(km.CircuitBuilder, "mr"))
print("angle_words:", words12.shape, "range", words12.min(), words12.max())

## 2. The cap is a product, not a width

Each `ANGLE_k` circuit must satisfy `num_qubits <= 128`. Its width is

$$Q = \underbrace{12}_{\text{selection: } \lceil\log_2 3976\rceil}
    + \underbrace{J \cdot brot}_{\text{word registers}}
    + \underbrace{11}_{\text{tree ancillas } = 12-1}$$

The selection width is fixed by the row count and the contract rejects extra selection
bits; the ancillas are fixed by the tree. Only $J \cdot brot$ is free, so the cap binds the
**product** — and because the number of circuits is $K = \lceil 75/J \rceil$, the cheapest
family is the largest $J$ that still fits.

In [ ]:
def requantise(words, src_bits, dst_bits):
    """Move a word table to dst_bits, keeping theta/pi = word / 2**(b-1).

    Words are turn counts modulo one turn, so a rounded 4095 becomes 1024 and must wrap
    to 0; without the mask the caller gets a word one past the register width.
    """
    words = np.asarray(words, dtype=np.int64)
    mask = (1 << dst_bits) - 1
    if dst_bits >= src_bits:
        return (words << (dst_bits - src_bits)) & mask
    shift = src_bits - dst_bits
    return ((words + (1 << (shift - 1))) >> shift) & mask


def batch_deltas(words, batch_size):
    """delta[0] = batch[0]; delta[k] = batch[k] XOR batch[k-1]. Shape (K, A, J)."""
    rows, angles = words.shape
    k = max(1, -(-angles // batch_size))
    padded = np.zeros((rows, k * batch_size), dtype=np.int64)
    padded[:, :angles] = words
    batches = padded.reshape(rows, k, batch_size).transpose(1, 0, 2)
    deltas = batches.copy()
    deltas[1:] ^= batches[:-1]
    return deltas


def build_delta_circuit(delta, brot):
    """One XOR oracle: a segment-tree QROM over the selection register."""
    rows, J = delta.shape
    nbits = (rows - 1).bit_length()
    b = km.CircuitBuilder()
    sel = b.create_quantum_register(nbits, C.SELECTION_REGISTER)
    words = [b.create_quantum_register(brot, C.word_register(j)) for j in range(J)]
    anc = b.alloc_qubits(nbits - 1) if nbits > 1 else []
    bits = [sel[i] for i in reversed(range(nbits))]

    def load(row, ctl):
        for j in range(J):
            w = int(delta[row, j])
            for bit in reversed(range(brot)):
                if (w >> bit) & 1:
                    b.cx(ctl, words[j][bit])

    def node(ctl, level, lo, hi):
        if lo >= rows:
            return
        if hi <= rows and (hi - lo == 1 or np.all(delta[lo:hi] == delta[lo])):
            load(lo, ctl)
            return
        mid = (lo + hi) // 2
        if rows <= mid:
            node(ctl, level + 1, lo, mid)
            return
        bit, a = bits[level], anc[level - 1]
        b.x(bit); b.init_and(ctl, bit, a); b.x(bit)
        node(a, level + 1, lo, mid)
        b.cx(ctl, a)
        node(a, level + 1, mid, hi)
        b.del_and(ctl, bit, a)

    half = 1 << (nbits - 1)
    b.x(bits[0]); node(bits[0], 1, 0, half)
    b.x(bits[0]); node(bits[0], 1, half, 2 * half)
    return b.finish_circuit()


def build_family(words, brot, J):
    return tuple(build_delta_circuit(d, brot) for d in batch_deltas(words, J))

In [ ]:
SELECTION = (words12.shape[0] - 1).bit_length()
print(f"selection {SELECTION} qubits, tree nodes <= {words12.shape[0]-1}\n")
print(f"{'brot':>4} {'J':>3} {'K':>3} {'peak Q':>7} {'spare':>6} {'T/circuit':>10} {'T (2 passes)':>13}")
rows = []
for brot, J in ((12, 3), (12, 8), (10, 3), (10, 8), (10, 10)):
    fam = build_family(requantise(words12, 12, brot), brot, J)
    costs = C.circuit_costs(fam)
    peak = max(c["num_qubits"] for c in costs)
    t2 = 2 * sum(c["max_magic"] for c in costs)
    rows.append(dict(brot=brot, J=J, K=len(fam), peak=peak, spare=128 - peak, T2=t2))
    print(f"{brot:4d} {J:3d} {len(fam):3d} {peak:7d} {128-peak:6d} {costs[0]['max_magic']:10d} {t2:13d}")

### Reading the sweep

Two rows do the work. `(brot 12, J 8)` needs 119 qubits and 10 circuits. `(brot 10, J 10)`
needs 123 and 8. Every row costs **3,974** Toffolis per circuit — one per tree node, minus
one node that early-break collapses — because the tree is over addresses, and nothing about
a word register changes how many nodes there are.

So the family cost is $2K \times 3{,}974$, and the whole game is $K$. Ten circuits → 79,480;
eight → 63,584. The 19.98% is bought with 4 qubits and two bits of precision, and both are
in surplus: 5 qubits of headroom and a quantisation step of $2\pi/2^{10}$.

## 3. Does it load the table? Ask the competition's checker

`recover_words_from_circuits` simulates every address, applies the family forward and in
reverse, and requires: the words after circuit $k$ equal batch $k$ on every shot, the
selection register unchanged, the workspace back to $|0\rangle$, no uncorrected phase, the
reverse pass clearing the registers, and the padding beyond 75 zero. It raises
`InvalidCircuits` with the reason otherwise, so a pass is a statement about the circuits and
not about this notebook's bookkeeping.

In [ ]:
BROT, J = 10, 10
words = requantise(words12, 12, BROT)
family = build_family(words, BROT, J)

recovered, layout, toffoli = C.recover_words_from_circuits(
    family, num_vectors=words.shape[0], num_angles=words.shape[1],
    shots_per_address=2, seed=20261102)

print("register layout     : brot", layout.brot, "| J", layout.batch_size,
      "| selection", len(layout.selection))
print("words recovered     :", "EXACT" if np.array_equal(recovered, words) else "MISMATCH")
print("words compared      :", recovered.size)
print("toffoli, fwd+rev    :", int(round(toffoli)))
print("peak qubits         :", max(c["num_qubits"] for c in C.circuit_costs(family)))
print("reaction depth      :", max(c["reaction_depth"] for c in C.circuit_costs(family)))

## 4. Idea that was measured and died: 9-bit delta payloads

If every angle satisfied $\cos\theta > 0$ then $\theta \in (-\pi/2, \pi/2)$, and in turn
`bit 9 == bit 8` for a 10-bit word — so a delta bank could store 9 bits and rebuild the tenth
by copying, saving $3 \times 3 \times 1 = 9$ qubits, which is exactly the margin the banked
designs need. It is a clean argument and it does not survive the data.

In [ ]:
def sign_invariant_fraction(w, brot):
    return float((((w >> (brot - 1)) & 1) == ((w >> (brot - 2)) & 1)).mean())

for brot in (12, 10):
    w = requantise(words12, 12, brot)
    print(f"brot={brot:2d}: bit{brot-1} == bit{brot-2} on {sign_invariant_fraction(w, brot)*100:6.2f}% "
          f"of {w.size} words")

w10 = requantise(words12, 12, 10)
theta = w10 * (2 * np.pi / 2 ** BROT)
print(f"\ntheta/pi range: [{theta.min()/np.pi:.4f}, {theta.max()/np.pi:.4f}]  "
      "(the invariant's premise is |theta| < pi/2)")
print("words that break it:", int((~(((w10 >> 9) & 1) == ((w10 >> 8) & 1))).sum()))

**78.59% is a falsification, not a rounding hiccup.** This table is not a chain of Givens
rotations with positive cosine; its angles cover $\theta/\pi \in [0, 2)$, i.e. a full turn,
so the top bit is a genuine angle bit and 63,810 of 298,200 words need it stored. Any design
that budgets the 9 qubits back has to earn them somewhere else — the sweep above does that
instead, by buying circuit count with precision rather than payload with an invariant.

The lesson generalises: the sign argument was proven for the object it was stated about, and
then applied to a different object. A one-line empirical check would have caught it before it
reached a qubit budget.

## 5. What 10 bits actually costs

Precision is only free in `T`; it is not free in the answer. The step at `brot = 10` is
$2\pi/2^{10}$, and requantising from the table's 12 bits moves every angle by at most half of
that — measured below rather than assumed.

In [ ]:
old = words12 * (2 * np.pi / 2 ** 12)
new = w10 * (2 * np.pi / 2 ** 10)
delta = (new - old + np.pi) % (2 * np.pi) - np.pi        # circular: a word is a turn count
print(f"max |dtheta|  = {np.abs(delta).max():.3e} rad")
print(f"mean |dtheta| = {np.abs(delta).mean():.3e} rad")
print(f"step          = {2*np.pi/2**BROT:.3e} rad")
print(f"fraction within half a step: {(np.abs(delta) <= np.pi/2**BROT).mean()*100:.2f}%")

## 6. What is not measured here, and the one thing needed to finish

No score, no $\lambda_{\mathrm{eff}}$, no $E_{\mathrm{gap}}$: all three come from the pinned
FCIDUMP, which is not in this bundle and cannot be substituted — `check_fcidump` compares both
size and SHA-256 and has no override argument. With that file mounted, the remaining step is
one command:

```bash
python dfthcsos/verify.py SUBMISSION_DIR --tree-edges SUBMISSION_DIR/submission.npz
```

The family above is the circuits half of that submission. The state-preparation half is the
shipped `U`, `W`, `H2_I`, `H1_I`, `Hsym_lower` arrays, which are unchanged by any of this.

## 7. Where the next gain is

- **One ancilla per tree is impossible, not merely expensive.** Sharing a single ancilla
  across the 11 levels was tried and kickmix refuses to build it: the parent's control has to
  survive its whole left subtree, so a child writing that qubit makes the parent's control and
  the child's target the same qubit (`CCX q122 q9 q122`, reported 7,943 times). The 11
  ancillas are not overhead — they are what makes the tree a tree.
- **`J = 11` is reachable through precision instead.** At `brot = 9` the family is 122 qubits
  (6 spare), `K = 7`, 55,636 Toffolis over both passes — 12.5% below `brot = 10, J = 10` —
  and the competition's checker recovers all 298,200 words exactly. It costs another bit:
  step 1.227e-2 rad, max error 6.136e-3. Whether that bit is affordable is a $\lambda_{\rm eff}$
  question, and therefore an FCIDUMP question, which is why this stops at the measurement.
  The break-even is computable without the integrals and is the useful form of the question:
  since the score is $\lambda_{\rm eff} \cdot T \cdot Q$, the 9-bit family still wins while
  $\lambda_{\rm eff}(9)/\lambda_{\rm eff}(10) \ge 0.875$ — it may lose up to 12.5% of
  $\lambda_{\rm eff}$ and remain the cheaper submission. Measured angular penalty: max
  6.136e-3 rad, rms 3.5e-3, worst-case rotation perturbation $2\sqrt{2}\sin(\delta/2)$ =
  8.7e-3 in Frobenius norm.
- **At the same batch, the freed bit is worthless.** `brot = 9, J = 10` is 113 qubits and costs
  exactly 63584 Toffolis — identical to the 10-bit family, because `K` is 8 either way —
  while the angle error doubles (3.068e-03 → 6.136e-03 rad). The extra ancillas the
  freed qubits could buy do nothing: the tree needs `n-1 = 11` and extra ones remove no nodes.
  The bit only pays when it removes a circuit, which is the `J = 11` row above.
- **Early-break has nothing to collapse.** All 3,976 delta rows are distinct in every one of
  the 8 circuits — 31,808 loads, 31,808 distinct — so a tree built over the delta tables
  finds no repeated subtree to share. The 3,974 Toffolis per circuit is the padding-trimmed
  tree (3,976 rows in 4,096 leaves), not a data-dependent saving.
  The break now compares whole rows rather than cells, which is strictly stronger and is what
  the ladder prototype uses; `--break-audit` shows it finds **0** collapsible ranges in each
  of the 8 circuits, so the count above is unchanged.
- **A sublinear-$T$ read was built, and on this table it loses.** The ladder read computes
  $\sqrt{A}$ offset controls once and keeps them alive, then loads each block by `ccx`.
  Addressing falls 21.7x (3,974 $\to$ 183 Toffolis), but a load with two controls is a
  Toffoli, so at this table's 40.0 set bits per address the read costs 40.3x the tree
  (160,064 against 3,974 at `J = 10`; 64,115 at `J = 4`). Given the same early-break rule
  this tree uses, it wins only below 0.87 set bits per address -- 0.84 without the break,
  0.95 if the tree cannot collapse at all -- so the bracket is 42x to 46x below what the
  table carries, whichever end is right.
- **What decides those numbers is empty addresses, not repeated words.** Scattering every
  set bit to its own bit position, so no two rows are equal, moves nothing (tree 1,405 at
  0.10 and 3,744 at 1.00 either way): a sparse table has all-zero blocks whatever its
  words look like. Filling one bit into *every* address is the one regime where the
  ladder was cheaper by 8% (3,638 against 3,974); with the tree breaking on row equality that
  tree is 3,453 and the ladder is 5% dearer -- and 40 bits on every address, which is
  this table, is 40x past it. Any sparser table that would be cheap has empty addresses by
  construction, so the two conditions cannot be met at once.
- **On the shipped `J = 10` family that window closes.** 100 payload qubits leave only
  `low = 2` inside the cap, so the address tree costs 996 Toffolis instead of 183: the
  crossing drops to 0.56, and the one-bit-per-address table that was 8% cheaper at
  `J = 4, low = 5` is 26% dearer (4,765 against 3,770). The bracket for the family that
  actually ships is `[0.56, 0.75]`. Structured payloads are also closer than the
  uncollapsed read suggested -- 280 against 131 with eight runs, 20 against 14 with one,
  now that the ladder's own offset tree is built only where a load addresses it -- but
  they still lose, so the ladder loses however this table is arranged.
- **`low = 2` was not the split to choose.** Costing every split with its qubit count
  shows the cap admits only `low <= 2` at `J = 10`, and the best of those is `low = 0` --
  the tree again (3,486 Toffolis at 0.75 bits per address, 3,770 at one bit on every
  address, against 4,765 at `low = 2`). No in-cap split wins on the shipped family; the
  ladder's advantage first appears at `low = 3`, which needs 134 qubits. So the 0.56
  crossing describes a split that is not the best one available.
- **Three of the four layouts never pick the ladder.** `--low-matrix` walks all eight
  densities on all four payload layouts. At `J = 10` the best in-cap split is `low = 0` --
  the tree -- for per-address (386-3,770 Toffolis), blocked (89-122) and contiguous (15) at
  every density; only the random layout ever chooses `low = 2` (0.79-0.96x at 0.10-0.50).
  The same grid at `J = 4`, where `low = 5` does fit, moves the random window out to 0.75
  bits per address (0.41-0.89x) and changes nothing for the structured tables: their trees
  are already 14-131 Toffolis, less than the ladder's 183-Toffoli addressing. Structure is
  not sparsity with order -- it is the case the tree is near-optimal on.
- **A flat comparator table is worse on both axes.** At `low = 12`, where the block count
  exceeds the rows, the read degenerates to one comparator per address: 6,980 and 7,548
  Toffolis at 7,094 and 7,662 qubits, against 3,691 and 3,770 at 123 for the tree. That is
  58x the qubits for no fewer Toffolis, so no table here is worth asking the cap for.
- **The second pass cannot be replaced by measurement-based uncomputation.** This is not a
  gate-set limit. The checker applies the family in reverse and asserts after *every* step that
  the word registers hold the previous batch (`reverse ANGLE_k did not restore the ANGLE_{k-1}
  word registers`), and that the final step clears them. A measurement-based clear produces
  zeros, so it fails at the first `k > 0` however the qubits are measured; and the reported
  Toffoli count is the forward and reverse passes together divided by shots, so the factor of 2
  is structural rather than a bookkeeping choice. The pinned gate set would block it anyway:
  there is no Z-basis read, and `R` randomises the tracked phase.
  Nor can the two applications share an address tree. Between the forward and reverse use of
  one circuit nothing differs — the checker re-executes the same bytes — so there is no second
  payload to fuse; the `T` charge is exactly `2 * sum(circuit Toffolis)`
  (2 × 31792 = 63584), and a family one circuit short is rejected
  against `ceil(75/J)` rather than completed by the sweep.


In [ ]:
out = pathlib.Path("submission"); out.mkdir(exist_ok=True)
for k, circuit in enumerate(family):
    circuit.to_file(C.kmb_path(out, k), format="kmb")
import shutil
shutil.copy(npz, out / "submission.npz")

import numpy as _np
with _np.load(npz, allow_pickle=False) as _z:
    _tree = _np.array(_z["tree_edges"], dtype=_np.int64)
digest = C.submission_sha256(
    [C.read_circuit(C.kmb_path(out, k), format="kmb") for k in range(len(family))],
    out / "submission.npz", _tree)
print("wrote", len(family), "circuits ->", out)
print("submission sha256:", digest)
print("fcidump present  :", (bundle / "verifiers" / "dfthcsos" / "FeMoCo76o113e.fcidump").is_file())

The digest in the last line is the value `verify.py` seeds its simulator from, so it is a
checkpoint: re-running the builder must reproduce it byte for byte. The `fcidump present:
False` is the honest state of this bundle and the only thing standing between these circuits
and a scored submission.